In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
import unidecode
import pulp

In [ ]:
# add comments to show where raw fbref html came from (urls)

In [12]:
df_fixtures_2324 = pd.read_html("fbref_fixtures_2324.html")[0]
df_fixtures_2324 = df_fixtures_2324.dropna(subset='Wk')
df_fixtures_2324 = df_fixtures_2324[df_fixtures_2324['xG'] != 'xG']
df_fixtures_2324['Wk'] = pd.to_numeric(df_fixtures_2324['Wk'])
df_fixtures_2324 = df_fixtures_2324[df_fixtures_2324['Wk'] <= 19]

df_fixtures_2425 = pd.read_html("fbref_fixtures_2425.html")[0]
df_fixtures_2425 = df_fixtures_2425.dropna(subset='Wk')
df_fixtures_2425 = df_fixtures_2425.dropna(subset='Score')
df_fixtures_2425 = df_fixtures_2425[df_fixtures_2425['xG'] != 'xG']

In [15]:
df_fixtures = pd.concat([df_fixtures_2324, df_fixtures_2425])

In [16]:
df_fixtures['xG'] = pd.to_numeric(df_fixtures['xG'])
df_fixtures['xG.1'] = pd.to_numeric(df_fixtures['xG.1'])

In [17]:
filtered_home_xg = df_fixtures[df_fixtures['xG'] < 1]
filtered_away_xg = df_fixtures[df_fixtures['xG.1'] < 1]

count_playing_away = filtered_home_xg['Away'].value_counts()
count_playing_home = filtered_away_xg['Home'].value_counts()

total_counts = count_playing_home.add(count_playing_away, fill_value=0)

In [18]:
df_total_counts = total_counts.reset_index()
df_total_counts.columns = ['Squad', 'xClean']

In [25]:
df_standard_2324 = pd.read_html("fbref_standard_2324.html")[0]
df_standard_2324.columns = df_standard_2324.columns.droplevel(0)
df_standard_2324.to_csv("fbref_standard_2324_raw.csv", index=False)

In [23]:
df_standard_2425 = pd.read_html("fbref_standard_2425.html")[0]
df_standard_2425.columns = df_standard_2425.columns.droplevel(0)
df_standard_2425.to_csv("fbref_standard_2425_raw.csv", index=False)

In [28]:
# filter out duplicate columns in google sheets

In [29]:
df_standard_2324 = pd.read_csv("fbref_standard_2324_filtered.csv")
df_standard_2425 = pd.read_csv("fbref_standard_2425_filtered.csv")

In [30]:
df_standard_2324 = df_standard_2324[df_standard_2324['Rk'] != 'Rk']
df_standard_2425 = df_standard_2425[df_standard_2425['Rk'] != 'Rk']

In [44]:
df_standard_2324 = df_standard_2324[['Player', '90s', 'npxG', 'xAG']]
df_standard_2425 = df_standard_2425[['Player', 'Squad', '90s', 'npxG', 'xAG']]

In [76]:
df_standard_2324[df_standard_2324['Player'] == "Cole Palmer"]

,Player,90s,npxG,xAG
430,Cole Palmer,29,11.1,11.1
431,Cole Palmer,0.1,0,0


In [75]:
df = df_standard_2324.merge(df_standard_2425, on="Player", how="inner")

In [46]:
df['npxG_x'] = pd.to_numeric(df['npxG_x'])
df['xAG_x'] = pd.to_numeric(df['xAG_x'])
df['90s_x'] = pd.to_numeric(df['90s_x'])

df['npxG_y'] = pd.to_numeric(df['npxG_y'])
df['xAG_y'] = pd.to_numeric(df['xAG_y'])
df['90s_y'] = pd.to_numeric(df['90s_y'])

In [52]:
df['90s'] = df['90s_y'] * 2
df['npxG'] = df['npxG_x']/2 + df['npxG_y']
df['xAG'] = df['xAG_x']/2 + df['xAG_y']

In [74]:
df[df['Player'] == "Cole Palmer"]

,Player,90s_x,npxG_x,xAG_x,Squad,90s_y,npxG_y,xAG_y,90s,npxG,xAG,xClean
243,Cole Palmer,29.0,11.1,11.1,Chelsea,17.4,8.3,6.6,34.8,13.85,12.15,9.0
244,Cole Palmer,0.1,0.0,0.0,Chelsea,17.4,8.3,6.6,34.8,8.30,6.60,9.0


In [54]:
df = df.merge(df_total_counts, how='left', on='Squad')
df['xClean'] = df['xClean'].fillna(0)

In [55]:
df_fpl_data = pd.read_csv("https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/cleaned_players.csv")

In [56]:
df_fpl_data.to_csv("current_prices_positions.csv", index=False)

In [57]:
df1 = df[["Player", "npxG"]]

df2 = df_fpl_data[['first_name', 'second_name', 'element_type', 'now_cost']]

# Function to preprocess names (remove accents, lowercase)
def preprocess_name(name):
    return unidecode.unidecode(name).lower()

df1['processed_name'] = df1['Player'].apply(preprocess_name)
df2['processed_name'] = (df2['first_name'] + ' ' + df2['second_name']).apply(preprocess_name)

# Fuzzy matching
def match_names_with_score(row, choices, scorer=fuzz.token_sort_ratio, threshold=60):
    match, score, _ = process.extractOne(row['processed_name'], choices, scorer=scorer)
    if score >= threshold:
        return match, score
    else:
        return None, None

# Perform the fuzzy matching
choices = df2['processed_name'].tolist()
df1['matched_name'], df1['score'] = zip(*df1.apply(match_names_with_score, axis=1, choices=choices))

# Merge DataFrames
merged_df = pd.merge(df1, df2, left_on='matched_name', right_on='processed_name', how='left')

# Drop intermediate columns if needed
merged_df = merged_df.drop(columns=['processed_name_x', 'processed_name_y'])

C:\Users\Peter\AppData\Local\Temp\ipykernel_4108\3738523934.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['processed_name'] = df1['Player'].apply(preprocess_name)
C:\Users\Peter\AppData\Local\Temp\ipykernel_4108\3738523934.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['processed_name'] = (df2['first_name'] + ' ' + df2['second_name']).apply(preprocess_name)
C:\Users\Peter\AppData\Local\Temp\ipykernel_4108\3738523934.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of

In [58]:
merged_df[merged_df['npxG'] >= 1].to_clipboard() # do some manual cleaning 'off-screen'

In [59]:
merged_df[merged_df['npxG'] >= 1].to_csv("clean_fpl_data.csv")

In [60]:
df_cleaned = pd.read_csv("clean_fpl_data.csv")

In [61]:
df_merged = df[['Player', 'Squad', 'npxG', 'xAG', 'xClean', '90s']].merge(df_cleaned[['Player', 'first_name', 'second_name', 'element_type', 'now_cost']])
df_merged = df_merged.rename(columns={"Player": "Name", "Squad": "Club", "element_type": "Position", "now_cost": "Cost"})

In [63]:
flat_pen_taker_bonus = (99 + 106 + 81 + 106 + 92) / 20 / 5 # number of penalties per season / 20 (for each team) averaged over last 5 seasons
top_bottom_diff = 1.2 # https://www.perplexity.ai/search/number-of-penalties-from-teams-hMulbzGRRaugUOmQUGiNaQ#1 (top ten teams averaged around 6 penalties, 1 higher)
penalty_xg = 0.79 # (opta)
print(f"top ten pen taker bonus: {(flat_pen_taker_bonus + top_bottom_diff) * penalty_xg}")
print(f"bottom ten pen taker bonus: {(flat_pen_taker_bonus - top_bottom_diff) * penalty_xg}")

df_merged['xG'] = df_merged['npxG']
df_merged.loc[df_merged['second_name'] == 'Haaland', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Saka', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Salah', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Heung-min', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
# df_merged.loc[df_merged['second_name'] == 'Solanke-Mitchell', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Palmer', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Borges Fernandes', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Watkins', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Pedro', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Isak', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Mbeumo', 'xG'] += (flat_pen_taker_bonus + top_bottom_diff) * penalty_xg

df_merged.loc[df_merged['second_name'] == 'Unal', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Mateta', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Calvert-Lewin', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Muniz Carvalho', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Gibbs-White', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Paqueta', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg
df_merged.loc[df_merged['second_name'] == 'Sarabia', 'xG'] += (flat_pen_taker_bonus - top_bottom_diff) * penalty_xg

top ten pen taker bonus: 4.7716
bottom ten pen taker bonus: 2.8756


In [64]:
df_merged['points_for_goal'] = df_merged['Position'].apply(lambda x: 6 if x == 'DEF' else 5 if x == 'MID' else 4)
df_merged['points_for_clean'] = df_merged['Position'].apply(lambda x: 4 if x == 'DEF' else 1 if x == 'MID' else 0)

In [65]:
df_merged = df_merged.reset_index(drop=True)

In [66]:
# df_merged.loc[df_merged['second_name'] == 'Haaland', 'xG'] = 38.0
# df_merged.loc[df_merged['second_name'] == 'Salah', 'xG'] = 18.0

In [67]:
df_merged['xPoints'] = df_merged['xG'] * df_merged['points_for_goal'] + \
    df_merged['xAG'] * 3 + \
    df_merged['xClean'] * df_merged['points_for_clean'] * df_merged['90s'] / 38

In [70]:
df_merged.to_csv("temp.csv", index=False)

In [73]:
df_merged.to_clipboard(index=False)

In [103]:
df_merged = df_merged[~df_merged['first_name'].isna()]
df_merged = df_merged[df_merged['Position'] != 'GK']
df_merged = df_merged[df_merged['Name'] != 'Mohamed Salah']
df_merged = df_merged[df_merged['Name'] != 'Bukayo Saka']
df_merged = df_merged[df_merged['second_name'] != 'Gvardiol']
df_merged = df_merged[df_merged['second_name'] != 'Semenyo']

df_merged = df_merged.drop_duplicates(inplace=False)

In [108]:
df_merged.loc[df_merged['second_name'] == 'Díaz', 'Cost'] = 76.0

In [112]:
max_players = 4  # Total number of players
max_per_club = 3  # Max players per club

# Define possible formations
formations = {
    # '3-5-2': {'FWD': 2, 'MID': 5, 'DEF': 3, 'budget': 785},
    # '4-4-2': {'FWD': 2, 'MID': 4, 'DEF': 4, 'budget': 785},
    # '3-4-3': {'FWD': 3, 'MID': 4, 'DEF': 3, 'budget': 785},
    # '4-5-1': {'FWD': 1, 'MID': 5, 'DEF': 4, 'budget': 785},
    'non-nailed 4-4-2': {'FWD': 2, 'MID': 2, 'DEF': 0, 'budget': 338},
    'non-nailed 4-5-1': {'FWD': 1, 'MID': 3, 'DEF': 0, 'budget': 332}
}

best_total_xPoints = 0
best_formation = None
best_selected_players = []
best_captain = None
best_total_cost = 0
best_players_by_position = {}

# Iterate through each formation
for formation_name, max_per_position in formations.items():
    # Create a linear programming problem
    prob = pulp.LpProblem(f"Fantasy_Football_Team_Selection_{formation_name}", pulp.LpMaximize)

    # Decision variables: 1 if the player is selected, 0 otherwise
    x = pulp.LpVariable.dicts("x", df_merged['second_name'], cat="Binary")

    # Decision variables for captains: 1 if the player is selected as captain, 0 otherwise
    ##### c = pulp.LpVariable.dicts("c", df_merged['second_name'], cat="Binary")

    # Objective function: Maximize the total expected goals (xG), considering captaincy
    ##### prob += pulp.lpSum([df_merged.loc[i, 'xPoints'] * (x[df_merged.loc[i, 'second_name']] + c[df_merged.loc[i, 'second_name']]) for i in df_merged.index])
    prob += pulp.lpSum([df_merged.loc[i, 'xPoints'] * x[df_merged.loc[i, 'second_name']] for i in df_merged.index])

    # Constraint 1: Budget constraint
    prob += pulp.lpSum([df_merged.loc[i, 'Cost'] * x[df_merged.loc[i, 'second_name']] for i in df_merged.index]) <= max_per_position['budget']

    # Constraint 2: Position constraints
    for position, max_count in max_per_position.items():
        if position != 'budget':  # Skip the 'budget' key
            prob += pulp.lpSum([x[df_merged.loc[i, 'second_name']] for i in df_merged.index if df_merged.loc[i, 'Position'] == position]) <= max_count

    # Constraint 3: Total number of players
    prob += pulp.lpSum([x[df_merged.loc[i, 'second_name']] for i in df_merged.index]) == max_players

    # Constraint 4: Club constraints
    clubs = df_merged['Club'].unique()
    for club in clubs:
        prob += pulp.lpSum([x[df_merged.loc[i, 'second_name']] for i in df_merged.index if df_merged.loc[i, 'Club'] == club]) <= max_per_club

    # Constraint 5: Exactly one captain
    ##### prob += pulp.lpSum([c[df_merged.loc[i, 'second_name']] for i in df_merged.index]) == 1

    # Constraint 6: A player can only be captain if they are also in the team
    ##### for i in df_merged.index:
    #####     prob += c[df_merged.loc[i, 'second_name']] <= x[df_merged.loc[i, 'second_name']]

    # Solve the problem
    prob.solve()

    # Calculate total xG and cost for this formation
    total_xPoints = sum(df_merged.loc[i, 'xPoints'] for i in df_merged.index if x[df_merged.loc[i, 'second_name']].varValue == 1) + \
               sum(df_merged.loc[i, 'xPoints'] for i in df_merged.index if c[df_merged.loc[i, 'second_name']].varValue == 1)
    total_cost = sum(df_merged.loc[i, 'Cost'] for i in df_merged.index if x[df_merged.loc[i, 'second_name']].varValue == 1)

    # Check if this formation is better
    if total_xPoints > best_total_xPoints:
        best_total_xPoints = total_xPoints
        best_formation = formation_name
        best_selected_players = [df_merged.loc[i, 'second_name'] for i in df_merged.index if x[df_merged.loc[i, 'second_name']].varValue == 1]
        ##### best_captain = [df_merged.loc[i, 'second_name'] for i in df_merged.index if c[df_merged.loc[i, 'second_name']].varValue == 1][0]
        best_total_cost = total_cost
        
        # Store selected players by position
        best_players_by_position = {
            position: [df_merged.loc[i, 'second_name'] for i in df_merged.index if x[df_merged.loc[i, 'second_name']].varValue == 1 and df_merged.loc[i, 'Position'] == position]
            for position in ['FWD', 'MID', 'DEF']
        }

# Print the best formation and selected players
print(f"Best Formation: {best_formation}")
print("Selected Players:")
for player in best_selected_players:
    print(player)
##### print(f"\nCaptain: {best_captain}")
print(f"Total Expected Points (xPoints): {best_total_xPoints}")
print(f"Total Cost: {best_total_cost} million")

# Print the selected players by position
print("\nSelected Players by Position:")
for position, players in best_players_by_position.items():
    print(f"{position}s: {', '.join(players)}")

Best Formation: non-nailed 4-4-2
Selected Players:
Borges Fernandes
Gordon
Isak
Watkins
Total Expected Points (xPoints): 492.6177473684211
Total Cost: 336.0 million

Selected Players by Position:
FWDs: Isak, Watkins
MIDs: Borges Fernandes, Gordon
DEFs: 


In [113]:
df_merged.to_csv("all_players_points_forecasts.csv", index=False)

In [114]:
df_merged.sort_values('xPoints', ascending=False).head(40)

,Name,Club,npxG,xAG,xClean,90s,first_name,second_name,Position,Cost,xG,points_for_goal,points_for_clean,xPoints
168,Cole Palmer,Chelsea,13.85,12.15,9.0,34.8,Cole,Palmer,MID,113.0,18.6216,5,1,137.800105
91,Erling Haaland,Manchester City,24.75,3.45,19.0,38.0,Erling,Haaland,FWD,148.0,29.5216,4,0,128.436400
96,Son Heung-min,Tottenham,8.90,11.70,10.0,26.8,Son,Heung-min,MID,98.0,13.6716,5,1,110.510632
101,Alexander Isak,Newcastle Utd,17.30,4.75,16.0,28.6,Alexander,Isak,FWD,91.0,22.0716,4,0,102.536400
75,Bruno Fernandes,Manchester Utd,7.65,10.10,11.0,32.8,Bruno,Borges Fernandes,MID,84.0,12.4216,5,1,101.902737
220,Ollie Watkins,Aston Villa,16.40,5.25,16.0,26.2,Ollie,Watkins,FWD,88.0,21.1716,4,0,100.436400
133,Gabriel MagalhÃ£es,Arsenal,4.45,0.85,22.0,30.2,Gabriel,dos Santos Magalhães,DEF,63.0,4.4500,6,4,99.186842
189,William Saliba,Arsenal,2.70,0.60,22.0,32.6,William,Saliba,DEF,63.0,2.7000,6,4,93.494737
170,Cole Palmer,Chelsea,8.30,6.60,9.0,34.8,Cole,Palmer,MID,113.0,13.0716,5,1,93.400105
6,Trent Alexander-Arnold,Liverpool,1.85,8.90,17.0,29.2,Trent,Alexander-Arnold,DEF,71.0,1.8500,6,4,90.052632
